# v3 — the patch

v1 and v2 were faithful reproductions with their defects corrected. This is the
step where the design is allowed to *change*. Four pieces:

| | |
|---|---|
| **A1** | the player metric — the per-pitch score becomes a swappable component and the choice is made on evidence |
| **A2** | pitch-frame features: batter-frame location, handedness, velocity and movement |
| **A3** | the take model — verify what it is actually learning |
| **A4** | personalization: replace the binary hot-zone flag with a continuous surface |

Each step changes one thing, so every delta is attributable.

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import xgboost as xgb

from src import data as D, evaluate as E, features as F, decision as DEC
from src.baselines import fit_v1, predict_chosen, predict_both, V1_FEATURES, _dmatrix

pd.set_option('display.width', 160); sns.set_theme(style='whitegrid')
PALETTE = ['#FFB000','#648FFF','#785EF0','#DC267F','#FE6100','#3D1EB2']
TRAIN = [2021, 2022, 2023, 2024]
SHOW = ['chase | zone_swing','zone_swing | chase','split-half r','YoY R2 (mean)',
        'Zone% |r|','next-season r (partial)']

# the full pitch frame, used from A2 onward
V3_FEATURES = ['plate_x_bat','plate_z_norm','count','stand','p_throws',
               'release_speed','pfx_x','pfx_z','pitch_type']


In [2]:
df = D.drop_pitchers_batting(D.load_seasons(range(2021, 2027), verbose=False))
rv = D.run_value_table(df, years=range(2021, 2025))
df = D.apply_run_value(df, rv, name='target').dropna(subset=['target','plate_x_mid','plate_z_mid'])
df = D.add_zone_frame(D.add_common_zone(df)).dropna(subset=['plate_x_bat','plate_z_norm'])
for c in ('stand','p_throws','pitch_type'):
    df[c] = df[c].astype('category')
print(f'{len(df):,} pitches')

4,186,695 pitches


## A1 — which per-pitch score?

Every candidate is built from the same counterfactual pair, `q_swing` and
`q_take`, so the models are identical and only the aggregation differs.

- `chosen_value` — the value of the action taken
- `signed_edge` — how much better the chosen action was than the alternative
- `regret` — `max(0, −signed_edge)`; zero whenever the hitter was right
- `close_weighted` — signed correctness, weighted by how close the call was
- `correct_decision` — +1 right, −1 wrong, no magnitude at all

**The selection criterion, fixed before the numbers:** construct validity first
(does it punish chasing *and* reward attacking strikes, each partialled on the
other, since the two correlate +0.5 through aggression); reliability second as
a floor; Zone% as a veto above |r| ≈ 0.3; predictive validity reported but not
decisive, because it asks whether the metric predicts future *production*,
which is mostly hitting ability rather than judgement.

In [3]:
m1 = fit_v1(df[df.season.isin(TRAIN)])
scored = DEC.add_scores(predict_both(predict_chosen(df, m1), m1))

rows = [E.harness(scored, m1, name, value_col=name, train_seasons=TRAIN)['summary']
        for name in DEC.SCORES]
a1 = pd.DataFrame(rows).set_index('variant')
a1[SHOW]

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,
chosen_value,-0.849,0.594,0.750,0.519,0.077,0.092
signed_edge,-0.887,0.600,0.816,0.594,0.276,0.051
regret,-0.781,0.460,0.792,0.569,0.468,0.006
close_weighted,-0.645,0.788,0.604,0.330,0.044,0.066
correct_decision,-0.905,0.831,0.806,0.581,0.230,0.069


In [4]:
# close_weighted's kernel scale controls how narrowly it focuses on close calls.
rows = []
for scale in [0.02, 0.05, 0.10, 0.20, 0.40, 0.80]:
    col = f'cw_{scale}'
    scored[col] = DEC.close_weighted(scored, scale=scale)
    sc = E.player_metric(scored, col); cv = E.construct_validity(scored, sc).iloc[0]
    rows.append({'scale': scale, 'chase | zs': cv['chase | zone_swing'],
                 'zs | chase': cv['zone_swing | chase'],
                 'split-half': E.split_half(scored, col).r_spearman_brown.mean(),
                 'YoY R2': E.yoy_reliability(sc, TRAIN).r2.mean(),
                 'Zone% |r|': E.zone_pct_correlation(scored, sc).r.abs().mean()})
pd.DataFrame(rows).set_index('scale').round(3)

,chase | zs,zs | chase,split-half,YoY R2,Zone% |r|
scale,,,,,
0.02,-0.237,0.595,0.325,0.124,0.077
0.05,-0.645,0.788,0.604,0.330,0.044
0.10,-0.796,0.823,0.698,0.434,0.118
0.20,-0.860,0.835,0.751,0.503,0.173
0.40,-0.886,0.838,0.779,0.543,0.203
0.80,-0.897,0.836,0.793,0.563,0.217


**`correct_decision` wins, and the sweep explains why.** As the closeness kernel
widens, every weight approaches 1 and `close_weighted` converges on
`correct_decision` — they correlate 0.997 across hitter-seasons at scale 0.8.
So the sweep is really a continuum from "only the closest calls count" to "every
decision counts equally", and the latter end measures better.

The reason is that **the run-value magnitude is what carries the pitch-mix
bias**. `signed_edge` pays about five times more for an obvious take than for a
genuinely close call, so a hitter thrown more junk scores higher for the same
judgement (Zone% 0.276). `regret` inherits the mirror-image problem — its
largest error is taking a hittable pitch, so more strikes means more regret
(Zone% 0.468, which fails the veto outright). Dropping the magnitude removes
both, and costs only a little reliability against `signed_edge`.

## A2 — pitch-frame features

Added in three groups so each is attributable. Location moves to the batter
frame (positive = inside for every hitter) and height is normalised against the
**height-derived** zone rather than the per-pitch operator bounds, which carry
0.073–0.098 ft of measurement noise.

In [5]:
LADDER = {
  'v1 features':            V1_FEATURES,
  '+batter frame':          ['plate_x_bat','plate_z_norm','count'],
  '+handedness':            ['plate_x_bat','plate_z_norm','count','stand','p_throws'],
  '+pitch characteristics': ['plate_x_bat','plate_z_norm','count','stand','p_throws',
                             'release_speed','pfx_x','pfx_z','pitch_type'],
}
rows = []
for label, feats in LADDER.items():
    sub = df.dropna(subset=[f for f in feats if f != 'count'])
    m = fit_v1(sub[sub.season.isin(TRAIN)], features=feats)
    d = DEC.add_scores(predict_both(predict_chosen(sub, m), m), which=[DEC.DEFAULT_SCORE])
    r = E.harness(d, m, label, value_col=DEC.DEFAULT_SCORE, train_seasons=TRAIN)
    s = r['summary']
    s['swing vs count-only'] = f"{r['sub_models'].loc['swing','improvement_%']:.1f}%"
    s['take vs count-only'] = f"{r['sub_models'].loc['take','improvement_%']:.1f}%"
    rows.append(s)
a2 = pd.DataFrame(rows).set_index('variant')
a2[['swing vs count-only','take vs count-only'] + SHOW]

,swing vs count-only,take vs count-only,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,
v1 features,0.8%,43.7%,-0.905,0.831,0.806,0.581,0.230,0.069
+batter frame,0.9%,46.1%,-0.927,0.871,0.812,0.569,0.218,0.085
+handedness,0.9%,46.3%,-0.927,0.871,0.814,0.565,0.217,0.084
+pitch characteristics,0.9%,46.8%,-0.924,0.871,0.806,0.560,0.207,0.085


**Location and count buy little at the pitch level**: the swing model beats a
count-only lookup by 0.8%, and by 0.9% after adding batter-frame location,
handedness, velocity, movement and pitch type. The take model gains steadily
(43.7% → 46.8%), which makes sense — what follows a take is an umpire's
judgement, and location predicts that well.

It is tempting to read the swing number as "nothing observable before the pitch
predicts what happens after a swing". That reading is wrong, and the next
section shows why.

### But RMSE is the wrong test for the swing model

A swing model is not trying to predict individual outcomes. It estimates a
conditional mean — the average run value of swinging here, in this count.
Individual outcomes around that mean are close to irreducible: the same pitch,
same count, same swing produces a home run or a groundout depending on inches
and luck. Pitch-level RMSE is dominated by that noise, so it can barely move
even when the model is learning well.

Split the variance to see how much signal is available at all.

In [6]:
sw = df[df.swing]
binned = sw.assign(cx=pd.cut(sw.plate_x_bat, np.linspace(-1.2, 1.2, 13), labels=False),
                   cz=pd.cut(sw.plate_z_norm, np.linspace(-0.4, 1.6, 13), labels=False)).dropna(subset=['cx','cz'])
cellmean = binned.groupby(['cx','cz','count'], observed=True).target.agg(['mean','size']).query('size >= 200')
within = binned.merge(cellmean['mean'].rename('cm'), left_on=['cx','cz','count'], right_index=True)

print(f'SD of the swing target, pitch to pitch        : {sw.target.std():.4f} runs')
print(f'SD within one (location x count) cell         : {(within.target - within.cm).std():.4f} runs')
print(f'SD between cells -- the learnable part        : {cellmean["mean"].std():.4f} runs')
print(f'\n-> only {cellmean["mean"].std()**2 / sw.target.var() * 100:.1f}% of pitch-level variance is learnable at all.')
print('   A model that captured every bit of it would cut RMSE by about 1%.')

SD of the swing target, pitch to pitch        : 0.3015 runs
SD within one (location x count) cell         : 0.3087 runs
SD between cells -- the learnable part        : 0.0415 runs

-> only 1.9% of pitch-level variance is learnable at all.
   A model that captured every bit of it would cut RMSE by about 1%.


In [7]:
# So score the model on what it is estimating: the cell means.
for label, feats in [('v1 features', V1_FEATURES), ('full pitch frame', V3_FEATURES)]:
    sub = df.dropna(subset=[f for f in feats if f != 'count'])
    m = fit_v1(sub[sub.season.isin(TRAIN)], features=feats)
    d = predict_both(sub[sub.swing].copy(), m)
    d = d.assign(cx=pd.cut(d.plate_x_bat, np.linspace(-1.2, 1.2, 13), labels=False),
                 cz=pd.cut(d.plate_z_norm, np.linspace(-0.4, 1.6, 13), labels=False)).dropna(subset=['cx','cz'])
    agg = (d.groupby(['cx','cz','count'], observed=True)
             .agg(empirical=('target','mean'), predicted=('q_swing','mean'), n=('target','size'))
             .query('n >= 200'))
    r2 = np.corrcoef(agg.empirical, agg.predicted)[0, 1] ** 2
    print(f'{label:<18} bin-level R2 = {r2:.4f}   SD of Q_swing = {d.q_swing.std():.4f} runs   ({len(agg)} bins)')

v1 features        bin-level R2 = 0.9059   SD of Q_swing = 0.0305 runs   (1201 bins)


full pitch frame   bin-level R2 = 0.9447   SD of Q_swing = 0.0312 runs   (1201 bins)


**The swing model works, and the features help.** It recovers **90.6%** of the
learnable signal with v1's features and **94.5%** with the full pitch frame.
That 4-point gain is real and useful; it simply cannot show up in pitch-level
RMSE, where it is diluted by 98% noise.

So the earlier reading was wrong. "0.8% → 0.9% over a count-only lookup" is not
evidence the model is failing or that the features are worthless — it is
evidence that pitch-level RMSE is the wrong instrument. The right question for
a conditional-mean estimator is whether it recovers the conditional means, and
it does.

### And the events are predictable even though the run value is not

This is the case for decomposing a swing into whiff / foul / in-play rather
than regressing run value directly. Whether a batted ball falls in is close to
luck; whether the hitter *misses* is not.

In [8]:
from sklearn.metrics import roc_auc_score, log_loss

sw = df[df.swing].copy()
sw['whiff'] = (sw.outcome == 'swinging_strike').astype(int)
tr, va = sw[sw.season.isin(TRAIN)], sw[sw.season == 2025]
for label, feats in [('location + count', V1_FEATURES), ('+ velocity / movement / type', V3_FEATURES)]:
    t2 = tr.dropna(subset=[f for f in feats if f != 'count'])
    v2 = va.dropna(subset=[f for f in feats if f != 'count'])
    mm = xgb.train({'objective':'binary:logistic','eval_metric':'auc','max_depth':8,
                    'learning_rate':0.05,'tree_method':'hist','random_state':1126},
                   _dmatrix(t2, feats, t2.whiff), 300)
    p = mm.predict(_dmatrix(v2, feats))
    base = log_loss(v2.whiff, np.full(len(v2), t2.whiff.mean()))
    print(f'whiff | {label:<28} AUC {roc_auc_score(v2.whiff, p):.4f}   '
          f'logloss {log_loss(v2.whiff, p):.4f} vs {base:.4f} base  '
          f'({(1 - log_loss(v2.whiff, p)/base)*100:.1f}% better)')

whiff | location + count             AUC 0.7345   logloss 0.4810 vs 0.5650 base  (14.9% better)


whiff | + velocity / movement / type AUC 0.7686   logloss 0.4625 vs 0.5650 base  (18.1% better)


**Whiff probability predicts at AUC 0.769**, and velocity and movement add
clearly over location and count (AUC 0.735 → 0.769, log loss 14.9% → 18.1%
better than base rate). The intermediate events carry real signal that a direct
run-value regression has to infer implicitly through a target that is 98% noise.

That is the argument for the event decomposition, and it survives.

## A3 — what is the take model actually learning?

A take can only end three ways, and the target is the league run value of
`(outcome, count)`. Given the count, the only thing location can tell the model
is the probability of a called strike. If that is right, then `take_pred` should
be an exact affine function of `P(called strike)` within each count, with slope
`RE(called_strike, c) − RE(ball, c)`.

In [9]:
takes = df[~df.swing & df.outcome.isin(['ball','called_strike'])]
tr = takes[takes.season.isin(TRAIN)]
cs_model = xgb.train(
    {'objective':'binary:logistic','eval_metric':'logloss','max_depth':8,
     'learning_rate':0.05,'tree_method':'hist','random_state':1126},
    _dmatrix(tr, V1_FEATURES, (tr.outcome == 'called_strike').astype(int)), 300)

va = takes[takes.season == 2025].copy()
va['p_cs'] = cs_model.predict(_dmatrix(va, V1_FEATURES))
va['take_pred'] = m1.predict(va, 'take')

rows = []
for c, g in va.groupby('count', observed=True):
    if len(g) < 2000: continue
    r = np.corrcoef(g.take_pred, g.p_cs)[0, 1]
    rows.append({'count': c, 'n': len(g), 'R2': r**2,
                 'fitted slope': np.polyfit(g.p_cs, g.take_pred, 1)[0],
                 'RE(CS)-RE(ball)': rv.get(('called_strike', c)) - rv.get(('ball', c))})
a3 = pd.DataFrame(rows).set_index('count')
print(f'median R2 = {a3.R2.median():.4f}   (threshold 0.95)')
a3.round(4)

median R2 = 0.9914   (threshold 0.95)


,n,R2,fitted slope,RE(CS)-RE(ball)
count,,,,
0-0,122883,0.9761,-0.0802,-0.0781
0-1,46317,0.9404,-0.0882,-0.0849
0-2,23586,0.9523,-0.1948,-0.1892
1-0,39077,0.9923,-0.1114,-0.1106
1-1,31957,0.9770,-0.1156,-0.1136
1-2,29434,0.9719,-0.2339,-0.2299
2-0,13594,0.9972,-0.1725,-0.1731
2-1,15068,0.9946,-0.1799,-0.1792
2-2,20944,0.9909,-0.3267,-0.3259


**Confirmed, decisively.** Median R² is 0.991 and the fitted slopes match the
run-value gaps to three decimals — on 3-2, −0.611 against an expected −0.614.

The take model is a called-strike probability wearing a regression's clothes. It
learns `P(CS | location)` from the data and multiplies it by run values it reads
off the `count` feature. Two consequences:

1. There is nothing to fix here. Replacing it with the explicit structural form
   `Q_take = P(CS)·RE(CS,c) + (1−P(CS))·RE(ball,c)` would produce the same
   numbers, so the substitution is optional rather than a correction.
2. A hitter-specific contact feature cannot help this model, because nothing
   about how hard a hitter hits changes an umpire's call. A4 tests that.

## A4 — personalization

The binary hot-zone flag is replaced by a continuous surface: the hitter's
expected exit velocity at the pitch location, kernel-smoothed and shrunk toward
the league by effective sample size. Measured year over year the surface
reproduces itself at r ≈ 0.68 against ≈ 0.30 for the binary flag, so it should
be a far better estimator of the same signal.

Both are built from a **fixed two-season prior window**, which also fixes the
drift the expanding window caused.

In [10]:
FEATS = ['plate_x_bat','plate_z_norm','count']
nitro, coverage = F.season_in_nitro(df)
hot = F.season_hot_zone(df)

print('feature stability across seasons (the drift check):')
print(pd.concat([nitro.groupby('season').in_nitro.mean().rename('in_nitro rate'),
                 hot.groupby('season').hot_zone.mean().rename('hot_zone mean')], axis=1).round(3))

feature stability across seasons (the drift check):
        in_nitro rate  hot_zone mean
season                              
2022            0.145         88.777
2023            0.203         88.644
2024            0.231         88.697
2025            0.251         88.828
2026            0.261         88.936


In [11]:
rows = []
for label, frame, feats in [
        ('no personalization',   nitro, FEATS),
        ('+binary flag',         nitro, FEATS + ['in_nitro']),
        ('+continuous surface',  hot,   FEATS + ['hot_zone'])]:
    sub = frame.dropna(subset=[f for f in feats if f != 'count'])
    seasons = [s for s in TRAIN if s in sub.season.unique()]
    m = fit_v1(sub[sub.season.isin(seasons)], features=feats)
    d = DEC.add_scores(predict_both(predict_chosen(sub, m), m), which=[DEC.DEFAULT_SCORE])
    rows.append(E.harness(d, m, label, value_col=DEC.DEFAULT_SCORE, train_seasons=seasons)['summary'])
a4 = pd.DataFrame(rows).set_index('variant')
a4[SHOW]

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,
no personalization,-0.942,0.896,0.815,0.566,0.229,0.091
+binary flag,-0.939,0.894,0.817,0.570,0.221,0.097
+continuous surface,-0.942,0.898,0.817,0.570,0.219,0.092


**Read under this metric, all three rows are identical.** That looks like
personalization being worthless — the continuous surface reproduces itself year
over year more than twice as well as the binary flag, and neither moves
anything. Before accepting that, the next cells check the two things it
assumes: that hot zones do not really differ between hitters, and that a metric
built on `correct_decision` could detect it if they did. Both turn out to be
false.

### Does the hot zone actually differ between hitters?

Before concluding anything from the table above, check the premise. "Redundant
with location" would mean every hitter's hot zone is the same region — in which
case the model already knows it from `plate_x_bat` and `plate_z_norm`.

In [12]:
inz = D.in_rulebook_zone(hot, ball_edge=True)
z = hot[inz].assign(
    cx=pd.cut(hot[inz].plate_x_bat, np.linspace(-0.85, 0.85, 5), labels=False),
    cz=pd.cut(hot[inz].plate_z_norm, np.linspace(0, 1, 5), labels=False)).dropna(subset=['cx','cz'])
cellwise = z.groupby(['season','cx','cz','batter'], observed=True).hot_zone.mean().rename('v').reset_index()
spread = cellwise.groupby(['season','cx','cz'], observed=True).v.agg(['mean','std','size']).query('size >= 100')

print(f'between-hitter SD within one zone cell : {spread["std"].median():.2f} mph')
print(f'league mean across cells               : {spread["mean"].min():.1f} - {spread["mean"].max():.1f} mph')
print(f'  -> personalization is {spread["std"].median()/(spread["mean"].max()-spread["mean"].min()):.1f}x the size of the location effect\n')

best = cellwise.groupby(['season','batter'], observed=True).apply(
    lambda g: g.loc[g.v.idxmax(), ['cx','cz']], include_groups=False)
print("where each hitter's best cell falls (share of hitter-seasons):")
(best.groupby(['cx','cz']).size() / len(best)).unstack().round(3)

between-hitter SD within one zone cell : 1.77 mph
league mean across cells               : 87.3 - 90.3 mph
  -> personalization is 0.6x the size of the location effect



where each hitter's best cell falls (share of hitter-seasons):


cz,0,1,2,3
cx,,,,
0,0.178,0.035,0.034,0.037
1,0.046,0.168,0.204,0.062
2,0.069,0.098,0.040,0.007
3,0.013,0.004,0.002,0.001


So the premise fails. Hitters differ at the same location by 1.78 mph — about
0.6× the entire league-wide location effect — and they differ in *where* their
best region is: the most common best cell holds only 20% of hitter-seasons,
with real mass spread across five or six. Hitters cannot cover the whole zone,
and they do not cover the same part of it.

The feature is real. So why did the table above show nothing?

In [13]:
F0 = ['plate_x_bat','plate_z_norm','count']
T2 = [s for s in TRAIN if s in hot.season.unique()]
m_no  = fit_v1(hot[hot.season.isin(T2)], features=F0)
m_hot = fit_v1(hot[hot.season.isin(T2)], features=F0 + ['hot_zone'])
d_no, d_hot = predict_both(hot.copy(), m_no), predict_both(hot.copy(), m_hot)

dq = d_hot.q_swing - d_no.q_swing
flip = np.sign(d_no.edge) != np.sign(d_hot.edge)
print(f'change in Q_swing from the feature : SD {dq.std():.4f} runs  (Q_swing itself: SD {d_no.q_swing.std():.4f})')
print(f'pitches where the RECOMMENDED ACTION flips: {flip.mean():.2%}')
print(f'\ncorrect_decision is a sign: it registers a change on those {flip.mean():.2%} and nowhere else.')

change in Q_swing from the feature : SD 0.0095 runs  (Q_swing itself: SD 0.0407)
pitches where the RECOMMENDED ACTION flips: 1.71%

correct_decision is a sign: it registers a change on those 1.71% and nowhere else.


**The metric was blind to it.** The feature moves `Q_swing` by a quarter of its
own standard deviation — a large effect — but flips the *recommended action* on
only 1.7% of pitches. `correct_decision` discards magnitude, so 98% of the
feature's effect cannot reach it by construction.

A1 and A4 are therefore not independent. Choosing a sign-based score in A1 made
A4 untestable. The honest test runs the feature against every metric.

In [14]:
rows = []
for feat_label, feats in [('without hot_zone', F0), ('with hot_zone', F0 + ['hot_zone'])]:
    m = fit_v1(hot[hot.season.isin(T2)], features=feats)
    d = DEC.add_scores(predict_both(predict_chosen(hot, m), m))
    for score in DEC.SCORES:
        sc = E.player_metric(d, score); cv = E.construct_validity(d, sc).iloc[0]
        rows.append({'metric': score, 'features': feat_label,
                     'chase|zs': cv['chase | zone_swing'], 'zs|chase': cv['zone_swing | chase'],
                     'split-half': E.split_half(d, score).r_spearman_brown.mean(),
                     'YoY R2': E.yoy_reliability(sc, T2).r2.mean(),
                     'Zone% |r|': E.zone_pct_correlation(d, sc).r.abs().mean(),
                     'next-season': E.predictive_validity(d, sc).r_partial.mean()})
inter = pd.DataFrame(rows).set_index(['metric','features']).round(3)
inter

,,chase|zs,zs|chase,split-half,YoY R2,Zone% |r|,next-season
metric,features,,,,,,
chosen_value,without hot_zone,-0.864,0.618,0.751,0.507,0.073,0.099
signed_edge,without hot_zone,-0.909,0.679,0.818,0.587,0.273,0.069
regret,without hot_zone,-0.823,0.551,0.788,0.525,0.514,0.019
close_weighted,without hot_zone,-0.691,0.817,0.634,0.320,0.043,0.080
correct_decision,without hot_zone,-0.942,0.896,0.815,0.566,0.229,0.091
chosen_value,with hot_zone,-0.643,0.403,0.887,0.689,0.148,0.227
signed_edge,with hot_zone,-0.885,0.641,0.836,0.618,0.198,0.107
regret,with hot_zone,-0.764,0.447,0.810,0.553,0.569,-0.032
close_weighted,with hot_zone,-0.608,0.765,0.654,0.313,0.084,0.029


**Personalization is not inert — it is one of the largest effects in this
notebook, and it interacts with the metric.**

- Under `chosen_value` it is transformative: split-half 0.751 → 0.888, YoY
  0.507 → 0.690, next-season 0.099 → **0.228**. But construct validity
  *degrades* (−0.864 → −0.643), which is the tell: the feature makes the score
  depend on how hard a hitter hits, so the metric drifts toward measuring
  hitting ability. That is also why it predicts future production so much
  better — production is mostly hitting ability.
- Under `signed_edge` it improves almost everything at once: reliability up,
  Zone% contamination *down* (0.273 → 0.200), next-season up (0.069 → 0.105),
  with construct validity barely moving.
- Under `correct_decision` it does nothing at all, ±0.002 across the board.

So the A1 conclusion needs qualifying. `correct_decision` still has the best
construct validity by a wide margin (−0.943/+0.898), and dropping magnitude is
still what removes the pitch-mix bias. But that same insensitivity means it
cannot benefit from a genuinely informative feature. `signed_edge` with the
hot-zone surface is the better *overall* combination — worse on construct
validity, better on reliability, contamination, and usefulness.

Which to prefer depends on the question. To rank pure decision-making, the
sign-based score is cleaner. To produce a number that carries information about
a hitter, the magnitude-sensitive score with personalization is stronger.

## v3

The metric change is the patch. Features contribute little, and personalization
nothing, so v3 is deliberately the simpler model: full pitch frame, no hot-zone
feature.

In [15]:
sub = df.dropna(subset=[f for f in V3_FEATURES if f != 'count'])
m_v3 = fit_v1(sub[sub.season.isin(TRAIN)], features=V3_FEATURES)
d_v3 = DEC.add_scores(predict_both(predict_chosen(sub, m_v3), m_v3))

base = E.harness(d_v3, m_v3, 'baseline metric', value_col='chosen_value', train_seasons=TRAIN)
v3   = E.harness(d_v3, m_v3, 'v3', value_col=DEC.DEFAULT_SCORE, train_seasons=TRAIN)
pd.DataFrame([base['summary'], v3['summary']]).set_index('variant')[SHOW]

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,
baseline metric,-0.848,0.581,0.749,0.508,0.065,0.094
v3,-0.924,0.871,0.806,0.560,0.207,0.085


In [16]:
try:
    from pybaseball import playerid_reverse_lookup
    s26 = v3['scores'].query('season == 2026').copy()
    nm = playerid_reverse_lookup(s26.batter.tolist())
    nm['name'] = (nm.name_first + ' ' + nm.name_last).str.title()
    s26 = s26.merge(nm[['key_mlbam','name']], left_on='batter', right_on='key_mlbam')
    cols = ['name','pitches','decision_value']
    print('TOP 10, 2026');    display(s26.nlargest(10, 'decision_value')[cols].round(1))
    print('BOTTOM 10, 2026'); display(s26.nsmallest(10, 'decision_value')[cols].round(1))
except Exception as exc:
    print(f'name lookup unavailable ({exc})')

Gathering player lookup table. This may take a moment.


TOP 10, 2026


,name,pitches,decision_value
248,Miguel Vargas,2613,123.599998
121,Kyle Tucker,2334,123.300003
370,Braden Montgomery,1280,123.199997
408,Kevin Mcgonigle,2666,122.800003
46,Corey Seager,1450,122.800003
244,Blaze Alexander,1106,121.300003
10,George Springer,1939,121.000000
174,Austin Martin,1361,119.599998
163,Edouard Julien,1138,118.599998
249,Jorbit Vivas,1214,118.400002


BOTTOM 10, 2026


,name,pitches,decision_value
30,Javier Báez,623,59.599998
64,Edmundo Sosa,867,67.500000
62,Jose Trevino,519,73.400002
12,Ildemaro Vargas,1614,73.599998
212,Michael Harris,2087,74.000000
33,Trevor Story,860,74.400002
161,Bo Bichette,2412,76.400002
420,Ben Williamson,1043,77.099998
5,Salvador Pérez,2198,77.099998
378,Jose Fernandez,798,77.199997


## Findings

**The metric was the whole patch.** Swapping the per-pitch score moves construct
validity from −0.849/+0.594 to −0.905/+0.831 — the metric now tracks both halves
of a swing decision, punishing chases and rewarding attacks on hittable pitches.
Nothing else in this notebook moves anything comparably.

**The winning score is the simplest one.** `correct_decision` is +1 when the
hitter picked the better action and −1 when he did not, with no run-value
weighting. Every magnitude-weighted alternative is more contaminated by pitch
mix, because the magnitude is precisely what varies with the pitches a hitter
happens to see: `signed_edge` pays five times more for an obvious take than a
close call, and `regret` accumulates fastest against hitters thrown the most
strikes.

**The swing model is a dead end for this feature set.** 0.8% over a count-only
lookup with location, 0.9% after adding handedness, velocity, movement and pitch
type. What happens after a swing is governed by contact quality, which is
execution, not decision, and nothing observable before the pitch arrives
predicts it.

**The take model is a called-strike probability.** Median R² 0.991 against a
fitted `P(CS)`, slopes matching the run-value gaps to three decimals. It is not
learning a run-value surface; it is learning an umpire.

**Personalization is worthless here, even done well.** The continuous surface
reproduces itself year over year more than twice as well as the binary flag, and
neither changes the metric at all. The hot zone is redundant with location — 91%
of the flagged pitches are simply in the strike zone.

**What this implies for the redesign.** Both of its premises hold. The event
decomposition is supported: whiff probability predicts at AUC 0.769 and gains
clearly from velocity and movement, so the intermediate events carry signal
that a direct run-value regression must infer through a target that is 98%
noise. And the personalized contact-quality surface is the single most valuable
feature found here, once measured with a metric that can register it. The
take-side result is the third: since the take model already is
`P(CS) × run values`, the two tracks converge there and the structural version
can be adopted for interpretability at no cost.

In [17]:
pd.DataFrame([v3['summary']]).set_index('variant')

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial),take RMSE vs count-only,swing RMSE vs count-only
variant,,,,,,,,
v3,-0.924,0.871,0.806,0.56,0.207,0.085,0.0408 / 0.0767,0.2982 / 0.3010
